# GRAST — WP07.4 CoT SFT (kaggle)

SFT từ checkpoint TinyStories 2.000 updates đã xác minh SHA256. Chỉ nạp model weights; optimizer/scaler/scheduler/RNG/data cursor khởi tạo mới cho stage SFT. **Không phải resume pretraining.** Model/tokenizer giữ nguyên. Mặc định arithmetic1500 train/300 validation, không dùng test hoặc GSM8K benchmark. Prompt và padding không tính loss; target gồm reasoning/final/EOS.

Mặc định `SMOKE_ONLY=True` dừng4 updates. Recipe gốc500 updates vẫn giữ nguyên. Train dài: `SMOKE_ONLY=False`; để tiếp tục smoke/mất runtime, điền `RESUME` tới checkpoint **SFT** latest, không điền checkpoint pretraining vào RESUME. Luôn stream log từng dòng.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
REPO = Path("/kaggle/working/grast-wp07-sft-source")
REVISION = "db0da482c3516d99b3fa6c3b6e6f566a65bf2c2f"  # verified WP07.4 implementation
SMOKE_ONLY = True
RESUME = ""  # full SFT latest.pt; empty means start SFT from pretrained weights


def run_logged(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        bufsize=1,
        env=os.environ.copy(),
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            status = process.wait()
        except BaseException:
            import signal

            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
            raise
        if status:
            raise subprocess.CalledProcessError(status, command)


if not REPO.exists():
    run_logged(["git", "clone", "https://github.com/CuongBien/GRAST.git", REPO])
run_logged(["git", "fetch", "origin"], cwd=REPO)
run_logged(["git", "checkout", "--detach", REVISION], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "matplotlib>=3.9,<4"], cwd=REPO)
run_logged(["git", "rev-parse", "HEAD"], cwd=REPO)

In [ ]:
import torch

print("Python:", sys.version, "Torch:", torch.__version__, "CUDA:", torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; no silent CPU fallback")
PRECISION = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print(
    "Training uses cuda:0:",
    torch.cuda.get_device_name(0),
    "precision:",
    PRECISION,
    "visible GPUs:",
    torch.cuda.device_count(),
    flush=True,
)

## Input và chạy nền

Add Input: SFT bundle ZIP (hoặc layout đã giải nén) và output pretraining chứa `final.pt`. Bật Internet/GPU; **Save Version → Save & Run All** để chạy trong batch session sạch, không có upload/mount tương tác. Checkpoint nguồn tự tìm theo đúng SHA256 đã xác minh; nếu cần đặt `PRETRAIN_CHECKPOINT` thành đường dẫn cụ thể. Sau smoke, Add Input output smoke và điền đường dẫn `RESUME` dưới `/kaggle/input/`, rồi Save & Run All lại. Output mới lưu dưới `/kaggle/working/grast-wp07-sft/runs/`.


In [ ]:
import tempfile

from llm_scratch.data.datasets import file_sha256
from llm_scratch.data.sft_bundle import extract_bundle, verify_directory

BUNDLE_NAME = "grast-wp07-sft-math-78f779a7d367.zip"
BUNDLE_SHA256 = "100296172c05aff2bb99a4cc314cadf6dece5c65e0d7d7bb846a323794e7296e"
PRETRAIN_SHA256 = "27cf2d9cb9774b0023d4db09e2735fd666452f5a945468bc71722bb3c69b5309"


OUTPUT_ROOT = Path("/kaggle/working/grast-wp07-sft/runs")
INPUT_ROOT = Path("/kaggle/input")
archives = sorted(INPUT_ROOT.rglob(BUNDLE_NAME))
if len(archives) == 1:
    INPUTS = extract_bundle(
        archives[0],
        Path(tempfile.mkdtemp(prefix="grast-sft-")) / "verified",
        expected_sha256=BUNDLE_SHA256,
    )
else:
    layouts = []
    for path in INPUT_ROOT.rglob("bundle_manifest.json"):
        import json

        if json.loads(path.read_text()).get("scope") == "cot_sft_train_validation_only":
            layouts.append(path.parent)
    if len(layouts) != 1:
        raise ValueError("Add exactly one SFT input bundle ZIP or extracted layout")
    INPUTS = verify_directory(layouts[0])
PRETRAIN_CHECKPOINT = ""  # optional explicit /kaggle/input/.../final.pt
if not RESUME:
    candidates = (
        [Path(PRETRAIN_CHECKPOINT)] if PRETRAIN_CHECKPOINT else sorted(INPUT_ROOT.rglob("final.pt"))
    )
    matches = [p for p in candidates if p.is_file() and file_sha256(p) == PRETRAIN_SHA256]
    if not matches:
        raise FileNotFoundError("Add pretraining final.pt with the verified SHA256 as Kaggle Input")
    PRETRAIN_CHECKPOINT = matches[0]
print("Verified SFT inputs:", INPUTS, flush=True)

## Chạy

Recipe khởi đầu: model46,15M, max_length512 (không truncate), batch2/accumulation4, LR5e-5/warmup25,500 successful updates. Full validation và latest checkpoint mỗi25 updates; giữ mốc50/100/250/final500. Prompt và padding labels=-100; các examples tách theo batch, không concatenate. BF16 khi GPU hỗ trợ, ngược lại FP16; fresh warm-start được đổi runtime/precision, exact SFT resume phải giữ recipe/implementation/runtime tương thích. Nếu lỗi giữa update, resume từ latest hoàn tất trước đó.


In [ ]:
command = [
    sys.executable,
    "-u",
    "-m",
    "llm_scratch.training.sft",
    "--config",
    REPO / "configs/training/wp07_sft.toml",
    "--prepared-root",
    INPUTS / "sft_data",
    "--tokenizer",
    INPUTS / "tokenizer",
    "--device",
    "cuda",
    "--precision",
    PRECISION,
    "--output-root",
    OUTPUT_ROOT,
]
if SMOKE_ONLY:
    command += ["--stop-after", "4"]
if RESUME:
    if not Path(RESUME).is_file():
        raise FileNotFoundError(RESUME)
    command += ["--resume", RESUME]
else:
    command += ["--init-checkpoint", PRETRAIN_CHECKPOINT]
run_logged(command, cwd=REPO)

In [ ]:
import json
import zipfile

from IPython.display import Image, display

RUN = sorted(OUTPUT_ROOT.iterdir(), key=lambda p: p.name)[-1]
print("Run:", RUN, "Resume:", RUN / "checkpoints/latest.pt", flush=True)
report_path = RUN / "artifacts/sft_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(
        json.dumps(
            {k: report[k] for k in ("status", "step", "planned_steps", "tokens_optimized")},
            indent=2,
        ),
        flush=True,
    )
if (RUN / "artifacts/loss_curves.png").exists():
    display(Image(filename=str(RUN / "artifacts/loss_curves.png")))
SUMMARY = OUTPUT_ROOT.parent / (RUN.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.suffix not in {".pt", ".u32"}:
            archive.write(path, path.relative_to(RUN))
print("Summary:", SUMMARY, flush=True)
print("Keep latest/final/milestone weights for WP07.5; summary ZIP excludes weights.", flush=True)